In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load feature data
X_train = np.loadtxt("Train/X_train.txt")
y_train = np.loadtxt("Train/y_train.txt", dtype=int)
X_test  = np.loadtxt("Test/X_test.txt")
y_test  = np.loadtxt("Test/y_test.txt", dtype=int)

# 2. Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

# 3. SMOTE over-sampling on training set
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train_enc)
print(f"SMOTE: before = {X_train.shape[0]}, after = {X_resampled.shape[0]}")

# 4. Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_resampled, label=y_resampled)
dtest  = xgb.DMatrix(X_test,    label=y_test_enc)
dtrain_orig = xgb.DMatrix(X_train, label=y_train_enc)

# 5. Set parameters with regularization, sampling, and early stopping
params = {
    'objective': 'multi:softmax',
    'num_class': len(le.classes_),
    'eval_metric': 'mlogloss',
    'max_depth': 3,
    'eta': 0.1,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'lambda': 5,    # L2 regularization
    'alpha': 1      # L1 regularization
}
num_round = 500

# 6. Train with early stopping
watchlist = [(dtrain, 'train'), (dtest, 'eval')]
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=watchlist,
    early_stopping_rounds=50,
    verbose_eval=10
)

# 7a. On SMOTE-augmented training set
y_res_pred = bst.predict(dtrain).astype(int)
print(f"Train Acc (SMOTE data): {accuracy_score(y_resampled, y_res_pred)*100:.2f}%")

# Classification report for SMOTE training set
print("\nClassification Report (SMOTE training):")
print(classification_report(
    y_resampled,
    y_res_pred,
    target_names=[str(c) for c in le.classes_]
))

# Confusion matrix heatmap for SMOTE training set
cm_res = confusion_matrix(y_resampled, y_res_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm_res, annot=True, fmt='d', cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (SMOTE training)")
plt.show()

# 7b. On original training set
y_orig_pred = bst.predict(dtrain_orig).astype(int)
print(f"\nTrain Acc (original data): {accuracy_score(y_train_enc, y_orig_pred)*100:.2f}%")

# Classification report for original training set
print("\nClassification Report (Original training):")
print(classification_report(
    y_train_enc,
    y_orig_pred,
    target_names=[str(c) for c in le.classes_]
))

# Confusion matrix heatmap for original training set
cm_orig = confusion_matrix(y_train_enc, y_orig_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm_orig, annot=True, fmt='d', cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Original training)")
plt.show()

# 8. Classification report and confusion matrix
print("Classification Report (Test):")
print(classification_report(y_test_enc, y_test_pred, target_names=[str(c) for c in le.classes_]))

cm = confusion_matrix(y_test_enc, y_test_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

SMOTE: before = 7767, after = 17076
[0]	train-mlogloss:2.06751	eval-mlogloss:2.09444
[10]	train-mlogloss:0.77944	eval-mlogloss:0.90196
[20]	train-mlogloss:0.39960	eval-mlogloss:0.55836
[30]	train-mlogloss:0.23412	eval-mlogloss:0.39831
[40]	train-mlogloss:0.15086	eval-mlogloss:0.31914


In [14]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
import pandas as pd
# 1. Overall & Balanced Accuracy
overall_acc  = accuracy_score(y_test_enc, y_test_pred)
balanced_acc = balanced_accuracy_score(y_test_enc, y_test_pred)
print(f"Overall Accuracy : {overall_acc*100:.2f}%")
print(f"Balanced Accuracy: {balanced_acc*100:.2f}%\n")

# 2. Per-class Accuracy
cm = confusion_matrix(y_test_enc, y_test_pred)
per_class_acc = cm.diagonal() / cm.sum(axis=1)

df_acc = pd.DataFrame({
    "Class":            le.classes_,
    "Per‐class Accuracy": per_class_acc
})
print("Per‐class Accuracy:")
print(df_acc.to_string(index=False))

Overall Accuracy : 91.08%
Balanced Accuracy: 81.51%

Per‐class Accuracy:
 Class  Per‐class Accuracy
     1            0.967742
     2            0.910828
     3            0.840476
     4            0.887795
     5            0.919065
     6            0.998165
     7            0.782609
     8            0.900000
     9            0.781250
    10            0.720000
    11            0.591837
    12            0.481481
